# 21 - Проверка воспроизводимости результатов

## Назначение

Загружаются сохранённые чекпоинты и выполняется оценка на тестовой выборке.
Полученные значения сверяются с числами, заявленными в README (пункты 5, 6, 7.5, 8.1, 8.4).
Обучение с нуля не выполняется.

## Целевая аудитория

Рецензент, которому нужно за минуты убедиться, что числа в README воспроизводятся
из артефактов, а не являются результатом ручного подбора.

## Что проверяется

- наличие чекпоинтов и CSV-файлов с результатами;
- совпадение размеров train/val/test и тестовых таргетов с заявленными в README §3;
- воспроизводимость метрик бейзлайнов (Popularity, Recent-Popularity, Item-kNN);
- воспроизводимость NDCG/Recall/MAP базовой GPT-2 v2 (π_ref);
- воспроизводимость MAE/RMSE/R^2 reward-модели v2;
- воспроизводимость RM-reward, NDCG@10, Recall@10, MAP@10 для RL-политик R0, R2, R3;
- согласованность оценок независимого судьи (LightGBM) для π_ref и RL-политик.

## Что не проверяется

- воспроизводимость обучения с нуля;
- полная последовательная проходка 20 ноутбуков;
- пересчёт аналитического Pareto-фронта, свипов по β и λ, diversity-профиля,
  мультишаговой генерации, bootstrap-интервалов - соответствующие артефакты
  сохранены в `results/`, но в задачу ноутбука не входят;
- **пересчёт тренировочной reward-функции R3 (`R0 + λ*log π_ref`)**:
  её реализация отсутствует в `src/reward_functions.py`. Для R3 оцениваются
  метрики политики (NDCG/Recall/MAP) и «RM reward» через базовую RM -
  ровно так, как это сделано в README 8.4, где столбец «RM reward»
  для всех политик считается по базовой RM.

## Ограничения

- Результаты зависят от согласованности `item2idx` с тем словарём, который
  использовался при обучении. Если `build_item_vocab` недоступен,
  словарь восстанавливается как `sorted(train.item_id) + 2` (PAD=0, UNK=1).
  При сильном расхождении NDCG@10 базовой GPT-2 с 0.1304 ноутбук явно
  сигнализирует о вероятном mismatch словаря.
- В режиме подвыборки (`SUBSET=10_000`) крупные эффекты (RM reward +8% у R0)
  воспроизводятся, малые дельты (NDCG −0.3% у R3) - нет.
  Для малых дельт требуется полный test (`SUBSET=None`).

## Время выполнения

- подвыборка 10 000 таргетов - ~2–5 минут на CPU, ~1 минута на GPU;
- полный test (186 690 таргетов) - ~30–60 минут на CPU, ~5–10 минут на GPU.

## Порядок работы

1. Ячейки 1–6: настройка, проверка артефактов, диагностика API.
2. Ячейки 7–18: загрузка данных, словаря, чекпоинтов и выполнение проверок.
3. Ячейки 19–20: сводный отчёт и сохранение `repro_report.json`.

Если ячейка падает с `TypeError` или `AttributeError`, это почти наверняка
несовпадение сигнатуры с текущим `src/` - диагностическая ячейка печатает
сигнатуры всех используемых функций.

In [1]:
import os, sys, json, time, inspect, traceback
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch

# Корень репозитория
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for p in [ROOT, *ROOT.parents]:
        if (p / "src").exists() and (p / "data").exists():
            ROOT = p
            os.chdir(p)
            break

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

print(f"Корень: {ROOT}")
print(f"Torch:  {torch.__version__}, CUDA: {torch.cuda.is_available()}")

# Режим оценки
# None - полный test, int - подвыборка для быстрой проверки крупных эффектов.
SUBSET = 10_000

TOL_REL       = 0.02   # 2% для детерминированных метрик
TOL_REL_NOISY = 0.05   # 5% для шумных метрик (RM, оценка RL на подвыборке)

# Пути к артефактам
CKPTS = {
    "gpt2_v2":   "results/checkpoints_v2/best_model.pt",
    "rm_v2":     "results/checkpoints_rm_v2/best_model.pt",
    "r0_dir":    "results/checkpoints_rl_r0_beta2.0",
    "r2_dir":    "results/checkpoints_rl_r2_beta2.0",
    "r3_dir":    "results/checkpoints_rl_r3_lam0.03_beta2.0",
    "judge":     "results/checkpoints_judge/judge",   # без расширения
}
REPRO_DIR = Path("results/repro_cache")
REPRO_DIR.mkdir(parents=True, exist_ok=True)

# Ожидаемые значения, из README 5, 6, 7.5, 8.4
EXPECTED = {
    "baselines": {
        "Popularity":        {"NDCG@10": 0.1032, "Recall@10": 0.1726},
        "Recent-Popularity": {"NDCG@10": 0.1229, "Recall@10": 0.2230},
        "Item-kNN":          {"NDCG@10": 0.1091, "Recall@10": 0.1842},
    },
    "gpt2_v2": {
        "Recall@10": 0.2192, "NDCG@10": 0.1304, "MAP@10": 0.1030,
        "Recall@50": 0.3714, "NDCG@50": 0.1638, "MAP@50": 0.1101,
        "Recall@100": 0.4592, "NDCG@100": 0.1780, "MAP@100": 0.1113,
    },
    "rm_v2": {"MAE": 0.3308, "RMSE": 0.3816, "R2": 0.0609},
    "policy": {
        "base (π_ref, = gpt2_v2)": {"RM_reward_top10": 0.5092, "Judge_reward_top10": 0.4817,
                 "Mean_item_mean_top10": 0.4850,
                 "NDCG@10": 0.1304, "Recall@10": 0.2192, "MAP@10": 0.1030},
        "r0":   {"RM_reward_top10": 0.5502, "Judge_reward_top10": 0.5069,
                 "Mean_item_mean_top10": 0.5245,
                 "NDCG@10": 0.1290, "Recall@10": 0.2179, "MAP@10": 0.1016},
        "r2":   {"RM_reward_top10": 0.5011, "Judge_reward_top10": 0.4740,
                 "Mean_item_mean_top10": 0.4731,
                 "NDCG@10": 0.1305, "Recall@10": 0.2188, "MAP@10": 0.1032},
        "r3":   {"RM_reward_top10": 0.5444, "Judge_reward_top10": 0.5036,
                 "Mean_item_mean_top10": 0.5189,
                 "NDCG@10": 0.1301, "Recall@10": 0.2198, "MAP@10": 0.1024},
    },
}

print(f"Режим: {'ПОДВЫБОРКА ' + str(SUBSET) if SUBSET else 'ПОЛНЫЙ TEST'}")

Корень: c:\Users\Vladislav\.vscode\TLabExam
Torch:  2.14.0+cu130, CUDA: True
Режим: ПОДВЫБОРКА 10000


## 1. Наличие артефактов

Проверяется наличие всех чекпоинтов, на которые ссылается README 13.
Если чекпоинтов нет, воспроизводимость уровня «чекпоинт -> метрика» невозможна.

In [2]:
print("Чекпоинты:")

def _check_file(label, path, extensions=None):
    p = Path(path)
    if extensions is not None:
        found = any((p.parent / (p.name + ext)).exists() for ext in extensions)
        if found:
            sizes = [(p.parent / (p.name + ext)).stat().st_size / 1e6
                     for ext in extensions if (p.parent / (p.name + ext)).exists()]
            print(f"  {label:10s} {p}.* ({', '.join(f'{s:.1f} MB' for s in sizes)})")
            return True
        print(f"  {label:10s} {p}.* - ОТСУТСТВУЕТ")
        return False
    else:
        if p.is_file():
            print(f"  {label:10s} {p} ({p.stat().st_size / 1e6:.1f} MB)")
            return True
        print(f"  {label:10s} {p} - ОТСУТСТВУЕТ")
        return False

missing_ckpts = []
for name, path in [
    ("gpt2_v2", CKPTS["gpt2_v2"]),
    ("rm_v2",   CKPTS["rm_v2"]),
]:
    if not _check_file(name, path):
        missing_ckpts.append(name)

# Директории RL: любой разумный файл чекпоинта (на всякий случай)
def _find_rl_ckpt(dir_path):
    d = Path(dir_path)
    if not d.exists():
        return None
    for cand in ["best_model.pt", "final_model.pt", "step_500.pt",
                 "step_400.pt", "step_300.pt", "step_200.pt", "step_100.pt"]:
        if (d / cand).exists():
            return d / cand
    pts = sorted(d.glob("*.pt"))
    return pts[-1] if pts else None

RL_CKPTS = {}
for name in ["r0_dir", "r2_dir", "r3_dir"]:
    found = _find_rl_ckpt(CKPTS[name])
    RL_CKPTS[name] = found
    if found is not None:
        print(f" {name:10s} {found} ({found.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f" {name:10s} {CKPTS[name]}/ - чекпоинт не найден (секция будет пропущена)")

# Judge: ожидаются .txt и .pkl рядом
judge_ok = _check_file("judge", CKPTS["judge"], extensions=[".txt", ".pkl"])
if not judge_ok:
    print(" Judge-секция будет пропущена.")

CSV_FILES = [
    "results/baselines_v3_masked.csv",
    "results/final_v3_with_v2.csv",
    "results/full_test_reward_vs_relevance.csv",
    "results/judge_eval_full_test.csv",
    "results/rl_r3_results/final_comparison.csv",
    "results/bootstrap_results/bootstrap_summary.csv",
]
print("\nCSV-файлы с результатами (для сверки README):")
for path in CSV_FILES:
    print(f"  {'Найден' if Path(path).exists() else 'Не найден '} {path}")

Чекпоинты:
  gpt2_v2    results\checkpoints_v2\best_model.pt (9.3 MB)
  rm_v2      results\checkpoints_rm_v2\best_model.pt (17.1 MB)
 r0_dir     results\checkpoints_rl_r0_beta2.0\final_model.pt (9.3 MB)
 r2_dir     results\checkpoints_rl_r2_beta2.0\final_model.pt (9.3 MB)
 r3_dir     results\checkpoints_rl_r3_lam0.03_beta2.0\final_model.pt (9.3 MB)
  judge      results\checkpoints_judge\judge.* (0.1 MB, 30.9 MB)

CSV-файлы с результатами (для сверки README):
  Найден results/baselines_v3_masked.csv
  Найден results/final_v3_with_v2.csv
  Найден results/full_test_reward_vs_relevance.csv
  Найден results/judge_eval_full_test.csv
  Найден results/rl_r3_results/final_comparison.csv
  Найден results/bootstrap_results/bootstrap_summary.csv


## 2. Диагностика API `src/`

Перед вызовом функций печатаются их сигнатуры. Это нужно, чтобы при
несовпадении аргументов не ловить `TypeError` в середине вычислений:
адаптация сводится к правке одной-двух строк в ячейках ниже.

In [3]:
import importlib

MODULES = [
    "src.metrics", "src.baselines", "src.my_evaluate",
    "src.reward_model", "src.rm_train", "src.rm_index",
    "src.reward_functions", "src.judge", "src.model",
    "src.dataset"
]

LOADED = {}
for modname in MODULES:
    try:
        LOADED[modname] = importlib.import_module(modname)
        print(f"\n[{modname}]")
        funcs = []
        for a in dir(LOADED[modname]):
            if a.startswith("_"):
                continue
            obj = getattr(LOADED[modname], a)
            if callable(obj) and not inspect.isclass(obj):
                try:
                    funcs.append(f"  {a}{inspect.signature(obj)}")
                except (ValueError, TypeError):
                    funcs.append(f"  {a}(...)")
        for line in funcs[:12]:
            print(line)
        if len(funcs) > 12:
            print(f"  ... и ещё {len(funcs) - 12}")
    except ImportError as e:
        print(f"\n[{modname}]  Внимание:  import failed: {e}")
        LOADED[modname] = None


[src.metrics]
  average_precision_at_k(preds, target, k)
  build_context(train_df, val_df=None)
  build_eval_targets(split_df)
  evaluate_model(predict_fn, targets_df, context, k_values=(10, 50, 100), verbose=True)
  ndcg_at_k(preds, target, k)
  pad_predictions(predict_fn, fallback, total_k=100)
  recall_at_k(preds, target, k)
  wrap_with_mask(predict_fn)

[src.baselines]
  item_knn_baseline(train, top_k_similar=50, top_n=500, max_history=20)
  last_item_baseline(train, top_n=500)
  popularity_baseline(train, top_n=500)
  pure_last_item_baseline(train)
  recent_popularity_at_cutoff(train, cutoff, window_days=14, top_n=500)
  recent_popularity_baseline(train, window_days=14, top_n=200)

[src.my_evaluate]
  evaluate_gpt2_fast(predictor, targets_df, context, k=100, batch_size=256, exclude_seen=True)
  evaluate_recent_popularity(train_df, targets_df, context, cutoff, window_days=14, top_n=500, k_values=(10, 50, 100))

[src.reward_model]
  create_reward_model(vocab_size, init_from=None, *

## 3. Данные и контрольные суммы

Ожидаемые значения из README 3:

| Сплит | Ожидаемый размер |
|---|---:|
| train | 3 787 046 |
| val (после фильтра) | 371 021 |
| test (после фильтра) | 474 130 |
| test targets (пар user-item) | 186 690 |
| уникальных пользователей в test | 132 323 |

`data/processed/test_targets.parquet` не создаётся `src/data.py`, поэтому
таргеты строятся через `metrics.build_eval_targets(test)`.

In [4]:
t0 = time.time()

train = pd.read_parquet("data/processed/train.parquet")
val   = pd.read_parquet("data/processed/val.parquet")
test  = pd.read_parquet("data/processed/test.parquet")

from src.metrics import build_eval_targets, build_context

# Таргеты строятся через тот же протокол, что и в src/metrics.py
test_targets = build_eval_targets(test)
val_targets  = build_eval_targets(val)

# Границы сплитов для RMIndex
t1 = val['last_watch_dt'].min()
t2 = test['last_watch_dt'].min()

CHECKS_DATA = OrderedDict()
CHECKS_DATA["train rows == 3 787 046"]   = (len(train) == 3_787_046, len(train))
CHECKS_DATA["val rows == 371 021"]       = (len(val)   == 371_021,   len(val))
CHECKS_DATA["test rows == 474 130"]      = (len(test)  == 474_130,   len(test))
CHECKS_DATA["test targets == 186 690"]   = (len(test_targets) == 186_690, len(test_targets))
CHECKS_DATA["test users == 132 323"]     = (test_targets['user_id'].nunique() == 132_323,
                                            test_targets['user_id'].nunique())
CHECKS_DATA["val targets == 176 224"]    = (len(val_targets) == 176_224, len(val_targets))
CHECKS_DATA["val users == 118 727"]      = (val_targets['user_id'].nunique() == 118_727,
                                            val_targets['user_id'].nunique())

print(f"\nДанные загружены за {time.time() - t0:.1f} с")
print(f"t1 (граница train/val) = {t1}")
print(f"t2 (граница val/test)  = {t2}\n")
for name, (ok, actual) in CHECKS_DATA.items():
    print(f"  {'Успех' if ok else 'Провал'}  {name:40s}  факт = {actual:,}")

# Контекст для test = история в train + val
context_test = build_context(train, val)

# Подвыборка
if SUBSET:
    rng = np.random.RandomState(1)
    idx = rng.choice(len(test_targets), size=min(SUBSET, len(test_targets)), replace=False)
    test_targets_eval = test_targets.iloc[idx].reset_index(drop=True)
    print(f"\nПодвыборка тестовых таргетов: {len(test_targets_eval):,} (seed=1)")
else:
    test_targets_eval = test_targets
    print(f"\nПолный набор тестовых таргетов: {len(test_targets_eval):,}")


Данные загружены за 0.4 с
t1 (граница train/val) = 2021-07-24 00:00:00
t2 (граница val/test)  = 2021-08-04 00:00:00

  Успех  train rows == 3 787 046                   факт = 3,787,046
  Успех  val rows == 371 021                       факт = 371,021
  Успех  test rows == 474 130                      факт = 474,130
  Успех  test targets == 186 690                   факт = 186,690
  Успех  test users == 132 323                     факт = 132,323
  Успех  val targets == 176 224                    факт = 176,224
  Успех  val users == 118 727                      факт = 118,727

Подвыборка тестовых таргетов: 10,000 (seed=1)


### Словарь

In [5]:
# Восстановление словаря item2idx / idx2item

item2idx = None
idx2item = None
vocab_source = None

def _try_build_vocab():
    global item2idx, idx2item, vocab_source
    try:
        from src.dataset import build_item_vocab as biv
    except ImportError:
        try:
            from src.train import build_item_vocab as biv
        except ImportError:
            biv = None

    if biv is None:
        return False

    for arg_desc, arg in [
        ("train_df", train),
        ("train+val+test", pd.concat([train, val, test], ignore_index=True)),
        ("item_series", train['item_id']),
    ]:
        try:
            result = biv(arg)
        except Exception:
            continue
        if isinstance(result, tuple) and len(result) == 2:
            item2idx, idx2item = result
            vocab_source = f"build_item_vocab({arg_desc})"
            return True
        if isinstance(result, dict):
            item2idx = result
            idx2item = {v: k for k, v in result.items()}
            vocab_source = f"build_item_vocab({arg_desc})"
            return True
    return False

if not _try_build_vocab():
    train_items = sorted(train['item_id'].unique())
    item2idx = {int(item_id): i + 2 for i, item_id in enumerate(train_items)}
    idx2item = {i: item_id for item_id, i in item2idx.items()}
    vocab_source = "sorted(train.item_id) + 2 (fallback)"

vocab_size = max(item2idx.values()) + 1
print(f"Источник словаря: {vocab_source}")
print(f"Размер словаря: {vocab_size:,} (ожидается 14 937 по README 6)")
if vocab_size != 14_937:
    print(f" Размер не совпадает с README. Возможен mismatch словаря.")
    print(f" Если NDCG@10 базовой GPT-2 окажется сильно ниже ожидаемого -")
    print(f" причина, скорее всего, в этом.")

Источник словаря: build_item_vocab(train_df)
Размер словаря: 14,937 (ожидается 14 937 по README 6)


### Вспомогательные функции для проверок

In [6]:
RESULTS = []

def section(title):
    """Печатает заголовок блока с рамкой — только для читаемости вывода."""
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")


def check(section_name, metric, expected, actual, tol):
    """
    Регистрирует одну проверку воспроизводимости.

    Параметры:
        section_name — название блока (например, "baselines", "gpt2_v2")
        metric       — название метрики (например, "NDCG@10")
        expected     — ожидаемое значение из README
        actual       — полученное значение (или None, если не измерено)
        tol          — относительный допуск (0.02 = 2%)

    Возвращает:
        None. Результат дописывается в глобальный список RESULTS.
    """
    if actual is None:
        status = "Плохо"
        msg = f"{metric}: НЕ ИЗМЕРЕНО"
    else:
        rel = abs(actual - expected) / (abs(expected) + 1e-12)
        status = "Хорошо" if rel <= tol else "Очень плохо"
        msg = (f"{metric}: ожидалось {expected:.4f}, "
               f"получено {actual:.4f}  (Δ = {rel * 100:+.1f}%)")

    RESULTS.append(dict(
        section=section_name,
        metric=metric,
        expected=expected,
        actual=actual,
        tol=tol,
        status=status,
        msg=msg,
    ))
    print(f"  {status}  {msg}")

## 5. Бейзлайны

Ожидаемые значения из README §5 (полный test, протокол с маскировкой):

| Модель | Recall@10 | NDCG@10 |
|---|---:|---:|
| Popularity | 0.1726 | 0.1032 |
| Recent-Popularity | 0.2230 | 0.1229 |
| Item-kNN | 0.1842 | 0.1091 |

Бейзлайны возвращают `predict_fn(user_id, context_items, context_last_date)`,
совместимы с `metrics.evaluate_model` напрямую.

In [7]:
section("5. Бейзлайны")
t0 = time.time()

from src.baselines import (
    popularity_baseline,
    recent_popularity_at_cutoff,
    item_knn_baseline,
)
from src.metrics import evaluate_model

tol_bl = TOL_REL if SUBSET is None else 0.10

train_val = pd.concat([train, val], ignore_index=True)

baselines = {
    "Popularity":        popularity_baseline(train, top_n=500),
    "Recent-Popularity": recent_popularity_at_cutoff(train_val, cutoff=t2, window_days=7, top_n=500),
    "Item-kNN":          item_knn_baseline(train),
}

for name, predict_fn in baselines.items():
    try:
        metrics = evaluate_model(predict_fn, test_targets_eval, context_test,
                                 k_values=(10,), verbose=False)
    except Exception:
        print(f"  Внимание:  {name}: evaluation failed")
        traceback.print_exc(limit=3)
        continue
    check("baselines", f"{name} · NDCG@10",   EXPECTED["baselines"][name]["NDCG@10"],
          metrics.get("NDCG@10"),   tol_bl)
    check("baselines", f"{name} · Recall@10", EXPECTED["baselines"][name]["Recall@10"],
          metrics.get("Recall@10"), tol_bl)

print(f"\nБейзлайны: {time.time() - t0:.1f} с")


5. Бейзлайны
  Хорошо  Popularity · NDCG@10: ожидалось 0.1032, получено 0.1057  (Δ = +2.4%)
  Хорошо  Popularity · Recall@10: ожидалось 0.1726, получено 0.1754  (Δ = +1.6%)
  Хорошо  Recent-Popularity · NDCG@10: ожидалось 0.1229, получено 0.1260  (Δ = +2.6%)
  Хорошо  Recent-Popularity · Recall@10: ожидалось 0.2230, получено 0.2262  (Δ = +1.4%)
  Хорошо  Item-kNN · NDCG@10: ожидалось 0.1091, получено 0.1118  (Δ = +2.5%)
  Хорошо  Item-kNN · Recall@10: ожидалось 0.1842, получено 0.1900  (Δ = +3.1%)

Бейзлайны: 22.3 с


## 6. Загрузка GPT-2-политики и полная оценка

Одна функция загружает чекпоинт GPT-2 (базовый или RL — они сохраняются
как `state_dict` от `GPT2LMHeadModel` в `src/train.py` / `src/rl_train.py`)
и выполняет полный цикл:

1. `GPT2Predictor.predict_batch` — top-100 рекомендаций на пользователя;
2. `evaluate_model` — NDCG/Recall/MAP;
3. `RMReward` — «RM reward top-10» (это метрика README §8.4, считается
   по базовой RM независимо от того, на какой reward обучалась политика);
4. `JudgeModel.score_predictions_topk` — независимая оценка top-10;
5. `JudgeModel.mean_item_mean_topk` — средний `item_mean` рекомендованных.

### Вспомогательные функции

In [8]:
from src.model import create_model
from src.my_evaluate import GPT2Predictor, evaluate_gpt2_fast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

def load_gpt2(ckpt_path):
    """Загружает GPT2LMHeadModel из state_dict. vocab_size берётся из весов."""
    state = torch.load(ckpt_path, map_location="cpu")
    if "transformer.wte.weight" in state:
        vs = state["transformer.wte.weight"].shape[0]
    elif "wte.weight" in state:
        vs = state["wte.weight"].shape[0]
    else:
        vs = vocab_size  # fallback
    model = create_model(vocab_size=vs, n_layer=2, n_head=4, n_embd=128,
                         n_positions=50, dropout=0.2)
    model.load_state_dict(state)
    model.eval()
    return model

@torch.no_grad()
def rm_reward_topk(rm_reward_fn, preds_dict, context, K=10, batch_size=1024):
    """Средний RM-reward top-K по пользователям."""
    ctx_list, act_list = [], []
    for u, items in preds_dict.items():
        ctx_info = context.get(u)
        ctx = ctx_info["items"] if ctx_info else []
        for it in items[:K]:
            ctx_list.append(ctx)
            act_list.append(int(it))
    if not ctx_list:
        return 0.0
    rewards = []
    for i in range(0, len(ctx_list), batch_size):
        r = rm_reward_fn(ctx_list[i:i + batch_size], act_list[i:i + batch_size])
        rewards.append(r.cpu().numpy())
    rewards = np.concatenate(rewards)
    users = np.array([u for u, items in preds_dict.items() for _ in items[:K]])
    df = pd.DataFrame({"u": users, "r": rewards})
    return float(df.groupby("u")["r"].mean().mean())


def eval_policy(name, ckpt_path, label, expected_key, judge=None, rm_reward_fn=None):
    """Полный цикл оценки одной GPT-2-политики (базовой или RL)."""
    if ckpt_path is None or not Path(ckpt_path).exists():
        print(f"  Внимание:  {name}: чекпоинт отсутствует — секция пропущена")
        return None

    try:
        model = load_gpt2(ckpt_path)
        predictor = GPT2Predictor(model, item2idx, idx2item, max_len=50, device=device)
    except Exception:
        print(f"  Внимание:  {name}: не удалось загрузить чекпоинт")
        traceback.print_exc(limit=3)
        return None

    # 1. Top-100 рекомендаций
    try:
        preds_dict = evaluate_gpt2_fast(
            predictor, test_targets_eval, context_test, k=100,
            batch_size=256, exclude_seen=True,
        )
    except Exception:
        print(f"  Внимание:  {name}: inference упал")
        traceback.print_exc(limit=3)
        return None

    # 2. Метрики ранжирования
    def predict_fn(user_id, ctx_items, ctx_date):
        return preds_dict.get(user_id, [])

    try:
        metrics = evaluate_model(
            predict_fn, test_targets_eval, context_test,
            k_values=(10, 50, 100), verbose=False,
        )
    except Exception:
        print(f"  Внимание:  {name}: evaluate_model упал")
        traceback.print_exc(limit=3)
        metrics = {}

    result = {
        "NDCG@10":  metrics.get("NDCG@10"),
        "Recall@10": metrics.get("Recall@10"),
        "MAP@10":   metrics.get("MAP@10"),
        "NDCG@50":  metrics.get("NDCG@50"),
        "NDCG@100": metrics.get("NDCG@100"),
    }

    # 3. RM reward top-10 (метрика из 8.4, базовая RM для всех политик)
    if rm_reward_fn is not None:
        try:
            result["RM_reward_top10"] = rm_reward_topk(
                rm_reward_fn, preds_dict, context_test, K=10,
            )
        except Exception:
            print(f"  Внимание:  {name}: RM reward упал")
            traceback.print_exc(limit=3)
            result["RM_reward_top10"] = None

    # 4. Judge: независимая оценка
    if judge is not None:
        try:
            result["Judge_reward_top10"] = judge.score_predictions_topk(preds_dict, K=10)
            result["Mean_item_mean_top10"] = judge.mean_item_mean_topk(preds_dict, K=10)
        except Exception:
            print(f"  Внимание:  {name}: judge eval упал")
            traceback.print_exc(limit=3)

    # 5. Диагностика vocabulary mismatch (только для базовой)
    if expected_key == "base (π_ref, = gpt2_v2)":
        if result.get("NDCG@10") is not None and result["NDCG@10"] < 0.08:
            print(f"  Внимание:  {name}: NDCG@10 = {result['NDCG@10']:.4f} << 0.1304. "
                  f"Возможен mismatch словаря (item2idx).")

    # 6. Сверка с ожидаемыми значениями
    exp = EXPECTED["policy"][expected_key]
    tol_rank = TOL_REL if SUBSET is None else 0.05
    tol_reward = 0.03 if SUBSET is None else 0.05

    for key, tol in [
        ("NDCG@10", tol_rank), ("Recall@10", tol_rank), ("MAP@10", tol_rank),
        ("RM_reward_top10", tol_reward),
        ("Judge_reward_top10", tol_reward),
        ("Mean_item_mean_top10", tol_reward),
    ]:
        if exp.get(key) is None or result.get(key) is None:
            continue
        check(label, key, exp[key], result[key], tol)

    return result

Устройство: cuda


## 7. Базовая GPT-2 v2 (π_ref): 6 и 8.4

Сначала загружается базовая GPT-2 и выполняются проверки README §6
(NDCG/Recall/MAP на K = 10, 50, 100) и README 8.4 (RM reward, judge reward,
item_mean для базовой политики). Одна и та же модель используется ниже как
π_ref для сравнения с RL.

In [9]:
section("7. Базовая GPT-2 v2 (π_ref)")
t0 = time.time()

# RM-функция (R0) нужна для метрики «RM reward top-10» во всех политиках.
# RM загружается в 8 ниже, но RMReward должен быть создан ДО 7,
# потому что RM reward считается и для базовой политики.
# Поэтому сначала загружаем RM.

from src.reward_model import create_reward_model
from src.reward_functions import RMReward

rm = None
if Path(CKPTS["rm_v2"]).exists():
    try:
        rm_state = torch.load(CKPTS["rm_v2"], map_location="cpu")
        vs_rm = rm_state["backbone.wte.weight"].shape[0]
        rm = create_reward_model(vocab_size=vs_rm)
        rm.load_state_dict(rm_state)
        rm.eval()
        rm = rm.to(device)
        print(f"RM v2 загружена (vocab_size = {vs_rm})")
    except Exception:
        print("  Внимание:  RM v2: не удалось загрузить")
        traceback.print_exc(limit=3)
        rm = None

rm_reward_fn = RMReward(rm, item2idx, max_len=50, device=device) if rm is not None else None

# Judge
judge = None
if Path(CKPTS["judge"] + ".txt").exists() and Path(CKPTS["judge"] + ".pkl").exists():
    try:
        from src.judge import JudgeModel
        judge = JudgeModel.load(CKPTS["judge"])
        print("Judge загружен")
    except Exception:
        print("  Внимание:  Judge: не удалось загрузить")
        traceback.print_exc(limit=3)
        judge = None

# Оценка базовой GPT-2
base_metrics = eval_policy(
    name="GPT-2 v2 (π_ref)",
    ckpt_path=CKPTS["gpt2_v2"],
    label="base (π_ref, = gpt2_v2)",
    expected_key="base (π_ref, = gpt2_v2)",
    judge=judge,
    rm_reward_fn=rm_reward_fn,
)

# Дополнительно — метрики 6 на K = 50, 100
if base_metrics is not None:
    for k in (50, 100):
        for m in ("NDCG", "Recall", "MAP"):
            key = f"{m}@{k}"
            if key in EXPECTED["gpt2_v2"] and base_metrics.get(key) is not None:
                tol = TOL_REL if SUBSET is None else 0.05
                check("gpt2_v2", key, EXPECTED["gpt2_v2"][key],
                      base_metrics[key], tol)

print(f"\nБазовая GPT-2: {time.time() - t0:.1f} с")


7. Базовая GPT-2 v2 (π_ref)


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.


RM v2 загружена (vocab_size = 14937)
Judge загружен
Уникальных юзеров для оценки: 9,726


predict_batch:   0%|          | 0/38 [00:00<?, ?it/s]

  Хорошо  NDCG@10: ожидалось 0.1304, получено 0.1328  (Δ = +1.9%)
  Хорошо  Recall@10: ожидалось 0.2192, получено 0.2245  (Δ = +2.4%)
  Хорошо  MAP@10: ожидалось 0.1030, получено 0.1046  (Δ = +1.5%)
  Хорошо  RM_reward_top10: ожидалось 0.5092, получено 0.5115  (Δ = +0.5%)
  Хорошо  Judge_reward_top10: ожидалось 0.4817, получено 0.4803  (Δ = +0.3%)
  Хорошо  Mean_item_mean_top10: ожидалось 0.4850, получено 0.4861  (Δ = +0.2%)
  Хорошо  NDCG@50: ожидалось 0.1638, получено 0.1654  (Δ = +1.0%)
  Хорошо  NDCG@100: ожидалось 0.1780, получено 0.1790  (Δ = +0.6%)

Базовая GPT-2: 16.1 с


## 8. Reward Model v2

Ожидаемые значения из README 7.5 (полный test):

| Метрика | Значение |
|---|---:|
| MAE | 0.3308 |
| RMSE | 0.3816 |
| R^2 | 0.0609 |

RM оценивается через `RMDataset` + `DataLoader` + `evaluate_regression`
на тех же данных и с тем же словарём, что использовались при обучении RM.
`RMIndex` строится из `concat(train, val, test)` с границами `t1 = val.min`,
`t2 = test.min`.

In [10]:
section("8. Reward Model v2")
t0 = time.time()

if rm is None:
    print("  Внимание:  RM v2 не загружена — секция пропущена")
else:
    from src.rm_index import RMIndex, RMDataset
    from src.rm_train import evaluate_regression
    from torch.utils.data import DataLoader

    try:
        all_df = pd.concat([train, val, test], ignore_index=True)
        rm_index = RMIndex(all_df=all_df, item2idx=item2idx, t1=t1, t2=t2)
        test_ds = RMDataset(rm_index, rm_index.test_mask, max_len=50)
        test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)
    except Exception:
        print("  Внимание:  RMIndex / RMDataset упал")
        traceback.print_exc(limit=3)
        test_loader = None

    if test_loader is not None:
        try:
            rm_metrics = evaluate_regression(rm, test_loader, device)
        except Exception:
            print("  Внимание:  evaluate_regression упал")
            traceback.print_exc(limit=3)
            rm_metrics = None

        if rm_metrics:
            tol_rm = TOL_REL_NOISY if SUBSET is None else 0.10
            check("rm_v2", "MAE",  EXPECTED["rm_v2"]["MAE"],  rm_metrics.get("mae"),  tol_rm)
            check("rm_v2", "RMSE", EXPECTED["rm_v2"]["RMSE"], rm_metrics.get("rmse"), tol_rm)
            check("rm_v2", "R2",   EXPECTED["rm_v2"]["R2"],   rm_metrics.get("r2"),   tol_rm * 2)

print(f"\nRM v2: {time.time() - t0:.1f} с")


8. Reward Model v2
RMIndex: 4,632,197 событий
  train: 2,789,962
  val:   371,021
  test:  474,130
RMDataset: 474,130 примеров
  Хорошо  MAE: ожидалось 0.3308, получено 0.3308  (Δ = +0.0%)
  Хорошо  RMSE: ожидалось 0.3816, получено 0.3816  (Δ = +0.0%)
  Хорошо  R2: ожидалось 0.0609, получено 0.0609  (Δ = +0.1%)

RM v2: 28.5 с


## 9. RL-политики R0, R2, R3

Ожидаемые значения из README 8.4 (полный test):

| Политика | RM reward top-10 | Judge reward top-10 | mean item_mean | NDCG@10 | Recall@10 | MAP@10 |
|---|---:|---:|---:|---:|---:|---:|
| π_ref | 0.5092 | 0.4817 | 0.4850 | 0.1304 | 0.2192 | 0.1030 |
| R0 (β=2.0) | 0.5502 | 0.5069 | 0.5245 | 0.1290 | 0.2179 | 0.1016 |
| R2 (β=2.0) | 0.5011 | 0.4740 | 0.4731 | 0.1305 | 0.2188 | 0.1032 |
| R3 (λ=0.03, β=2.0) | 0.5444 | 0.5036 | 0.5189 | 0.1301 | 0.2198 | 0.1024 |

Крупные эффекты (RM reward +8% у R0) воспроизводятся и на подвыборке.
Малые дельты (NDCG −0.3% у R3) на подвыборке неразличимы — для них нужен
полный test (`SUBSET=None`).

**Про R3.** В `src/reward_functions.py` отсутствует реализация тренировочной
reward-функции R3 (`R0 + λ·log π_ref`). Воспроизводятся метрики политики
(NDCG/Recall/MAP) и «RM reward top-10» — последний считается по базовой RM
так же, как для остальных политик в README §8.4. Пересчёт тренировочной
reward-функции R3 в этот ноутбук не входит.

In [11]:
section("9. RL-политики")
t0 = time.time()

policies = [
    ("R0 (β=2.0)",         RL_CKPTS.get("r0_dir"), "r0"),
    ("R2 (β=2.0)",         RL_CKPTS.get("r2_dir"), "r2"),
    ("R3 (λ=0.03, β=2.0)", RL_CKPTS.get("r3_dir"), "r3"),
]

for name, ckpt, key in policies:
    eval_policy(
        name=name,
        ckpt_path=ckpt,
        label=key,
        expected_key=key,
        judge=judge,
        rm_reward_fn=rm_reward_fn,
    )

print(f"\nRL-политики: {time.time() - t0:.1f} с")


9. RL-политики
Уникальных юзеров для оценки: 9,726


predict_batch:   0%|          | 0/38 [00:00<?, ?it/s]

  Хорошо  NDCG@10: ожидалось 0.1290, получено 0.1324  (Δ = +2.6%)
  Хорошо  Recall@10: ожидалось 0.2179, получено 0.2244  (Δ = +3.0%)
  Хорошо  MAP@10: ожидалось 0.1016, получено 0.1040  (Δ = +2.4%)
  Хорошо  RM_reward_top10: ожидалось 0.5502, получено 0.5520  (Δ = +0.3%)
  Хорошо  Judge_reward_top10: ожидалось 0.5069, получено 0.5058  (Δ = +0.2%)
  Хорошо  Mean_item_mean_top10: ожидалось 0.5245, получено 0.5256  (Δ = +0.2%)
Уникальных юзеров для оценки: 9,726


predict_batch:   0%|          | 0/38 [00:00<?, ?it/s]

  Хорошо  NDCG@10: ожидалось 0.1305, получено 0.1325  (Δ = +1.5%)
  Хорошо  Recall@10: ожидалось 0.2188, получено 0.2236  (Δ = +2.2%)
  Хорошо  MAP@10: ожидалось 0.1032, получено 0.1044  (Δ = +1.1%)
  Хорошо  RM_reward_top10: ожидалось 0.5011, получено 0.5039  (Δ = +0.6%)
  Хорошо  Judge_reward_top10: ожидалось 0.4740, получено 0.4729  (Δ = +0.2%)
  Хорошо  Mean_item_mean_top10: ожидалось 0.4731, получено 0.4747  (Δ = +0.3%)
Уникальных юзеров для оценки: 9,726


predict_batch:   0%|          | 0/38 [00:00<?, ?it/s]

  Хорошо  NDCG@10: ожидалось 0.1301, получено 0.1335  (Δ = +2.6%)
  Хорошо  Recall@10: ожидалось 0.2198, получено 0.2266  (Δ = +3.1%)
  Хорошо  MAP@10: ожидалось 0.1024, получено 0.1049  (Δ = +2.4%)
  Хорошо  RM_reward_top10: ожидалось 0.5444, получено 0.5465  (Δ = +0.4%)
  Хорошо  Judge_reward_top10: ожидалось 0.5036, получено 0.5024  (Δ = +0.2%)
  Хорошо  Mean_item_mean_top10: ожидалось 0.5189, получено 0.5202  (Δ = +0.2%)

RL-политики: 43.3 с


## 10. Сводный отчёт

Все выполненные проверки собраны в таблицу с колонками
`section`, `metric`, `expected`, `actual`, `status`. Отчёт сохраняется
в `results/repro_cache/repro_report.json`.

In [12]:
section("ИТОГОВЫЙ ОТЧЁТ О ВОСПРОИЗВОДИМОСТИ")

if len(RESULTS) == 0:
    print("Ни одна проверка не выполнена — см. ошибки в ячейках выше.")
else:
    df = pd.DataFrame(RESULTS)
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 200)
    print(df[["section", "metric", "expected", "actual", "status"]].to_string(index=False))

    n_pass = int((df["status"] == "Хорошо").sum())
    n_fail = int((df["status"] == "Очень плохо").sum())
    n_skip = int((df["status"] == "Плохо").sum())
    total  = len(df)

    print(f"\nИтого: Хорошо {n_pass}/{total}   Очень плохо {n_fail}/{total}   Плохо {n_skip}/{total}")

    if n_fail == 0 and n_skip == 0:
        verdict = "PASSED"
        print("\n ВОСПРОИЗВОДИМОСТЬ: PASSED — все измеренные метрики совпали с README.")
    elif n_fail == 0:
        verdict = "PARTIAL"
        print("\n ВОСПРОИЗВОДИМОСТЬ: PARTIAL — измеренные метрики совпали, "
              "часть проверок пропущена (нет чекпоинта или ошибка API).")
    else:
        verdict = "FAILED"
        print("\n ВОСПРОИЗВОДИМОСТЬ: FAILED — см. строки со статусом Очень плохо. "
              "Возможные причины: (а) README drift относительно текущего кода, "
              "(б) несовпадение API в этом ноутбуке, (в) повреждён чекпоинт, "
              "(г) mismatch словаря item2idx.")

    report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "subset": SUBSET,
        "verdict": verdict,
        "n_pass": n_pass, "n_fail": n_fail, "n_skip": n_skip,
        "total": total,
        "vocab_source": vocab_source,
        "vocab_size": vocab_size,
        "checks": df.to_dict(orient="records"),
    }
    out = REPRO_DIR / "repro_report.json"
    out.write_text(json.dumps(report, indent=2, default=str))
    print(f"\nОтчёт сохранён: {out}")


ИТОГОВЫЙ ОТЧЁТ О ВОСПРОИЗВОДИМОСТИ
                section                        metric  expected   actual status
              baselines          Popularity · NDCG@10    0.1032 0.105656 Хорошо
              baselines        Popularity · Recall@10    0.1726 0.175400 Хорошо
              baselines   Recent-Popularity · NDCG@10    0.1229 0.126039 Хорошо
              baselines Recent-Popularity · Recall@10    0.2230 0.226200 Хорошо
              baselines            Item-kNN · NDCG@10    0.1091 0.111813 Хорошо
              baselines          Item-kNN · Recall@10    0.1842 0.190000 Хорошо
base (π_ref, = gpt2_v2)                       NDCG@10    0.1304 0.132849 Хорошо
base (π_ref, = gpt2_v2)                     Recall@10    0.2192 0.224500 Хорошо
base (π_ref, = gpt2_v2)                        MAP@10    0.1030 0.104577 Хорошо
base (π_ref, = gpt2_v2)               RM_reward_top10    0.5092 0.511492 Хорошо
base (π_ref, = gpt2_v2)            Judge_reward_top10    0.4817 0.480336 Хорошо
base